In [2]:
# wide_format_dataset.py
import pandas as pd

# Assume df_long already exists from Template A or your own data
# If needed, load it:
# df_long = pd.read_csv('allsaints_uk_us_long_combined.csv', parse_dates=['period_start'])

# 1) Pivot to wide: example pivot for sales by region, age_group, gender, income
def pivot_to_wide(df, index_cols, column_cols, value_col, aggfunc='sum'):
    """
    General helper to pivot long to wide.
    index_cols: list of columns to keep as rows (e.g., country, period_start)
    column_cols: list of columns whose unique combinations form new columns
    value_col: the column with the values to place in the wide cells
    """
    df_wide = df.pivot_table(
        index=index_cols,
        columns=column_cols,
        values=value_col,
        aggfunc=aggfunc,
        fill_value=0
    )
    # Flatten column MultiIndex if needed
    if isinstance(df_wide.columns, pd.MultiIndex):
        df_wide.columns = ['_'.join(map(str, col)).strip() for col in df_wide.columns.values]
    df_wide = df_wide.reset_index()
    return df_wide

# 2) Example: wide format by country, period, region → sales
# Using df_long from Template A or your own dataset
# df_wide_region_sales = pivot_to_wide(df_long, index_cols=['country', 'period_start'], column_cols=['region'], value_col='sales')

# 3) Example: wide format by country, period, demographic combination → sales
# This could create many columns; only use if manageable
df_long = pd.read_csv('allsaints_uk_us_long_combined.csv', parse_dates=['period_start'])

df_wide_demo_sales = pivot_to_wide(
    df_long,
    index_cols=['country', 'period_start'],
    column_cols=['region', 'age_group', 'gender', 'income_level'],
    value_col='sales',
    aggfunc='sum'
)

df_wide_demo_sales.to_csv('allsaints_uk_us_wide_demo_sales.csv', index=False)

# 4) Wide format for overall metrics, e.g., sales + margins side by side
# You may want one wide per country, or combine
df_overall = (
    df_long.groupby(['country', 'period_start'])
           .agg(
               total_sales=('sales', 'sum'),
               avg_gross_margin_pct=('gross_margin_pct', 'mean'),
               avg_profit_margin_pct=('profit_margin_pct', 'mean'),
           )
           .reset_index()
)
# Pivot country into columns if desired
df_overall_wide = df_overall.pivot_table(
    index='period_start',
    columns='country',
    values=['total_sales', 'avg_gross_margin_pct', 'avg_profit_margin_pct']
).reset_index()
# Flatten columns
df_overall_wide.columns = ['_'.join(filter(None, col)).strip() for col in df_overall_wide.columns.values]
df_overall_wide.to_csv('allsaints_overall_wide_by_country.csv', index=False)